# 01 · TF-IDF — weighting words by how informative they are
Plain bag-of-words counts every word equally. **TF-IDF** downweights common words (the, is) and upweights rare, distinctive ones — so retrieval focuses on what actually distinguishes a document. This is the classic step between raw counts and BM25.

## 1. The idea
```
  TF  (term frequency)     = how often a word appears in THIS doc
  IDF (inverse doc freq)   = how RARE the word is across ALL docs
  TF-IDF = TF * IDF        = frequent-here AND rare-overall = distinctive
```
A word like 'refund' appearing in one doc scores high there; 'the' appearing everywhere scores near zero.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

docs = [
    "you can request a refund within 30 days",
    "refunds go back to your original payment method",
    "standard shipping takes three to five days",
    "track your package with the tracking number",
]
vec = TfidfVectorizer()
X = vec.fit_transform(docs)          # sparse TF-IDF matrix (docs x terms)
print("vocabulary:", vec.get_feature_names_out()[:12], "...")
print("matrix shape (docs, terms):", X.shape)

## 2. See the weights — 'refund' should outweigh 'days'

In [ ]:
import numpy as np
terms = vec.get_feature_names_out()
dense = X.toarray()
# show the top-weighted term in each doc
for i, row in enumerate(dense):
    top = terms[np.argmax(row)]
    print(f"doc {i}: top term = '{top}'  (weight {row.max():.3f})  | {docs[i][:40]}")

## 3. Retrieval with TF-IDF + cosine

In [ ]:
def search(query, k=3):
    qv = vec.transform([query])
    sims = cosine_similarity(qv, X).ravel()
    order = sims.argsort()[::-1][:k]
    return [(i, round(float(sims[i]),3), docs[i]) for i in order]

for i, s, t in search("how do I get my money refunded"):
    print(f"  doc {i}  sim={s}  {t}")

**Observe:** TF-IDF ranks the refund docs first because 'refund' is rare (high IDF) and present in those docs. Common words don't distract it. **Limitation (same as all keyword methods):** it still needs *shared words* — 'money refunded' matches 'refund' only via the shared token 'refund'; a pure synonym with no shared word (e.g. 'reimbursement') would be missed. That's what dense embeddings fix.

**TF-IDF vs BM25:** BM25 is TF-IDF plus saturation (extra occurrences matter less) and document-length normalization — a refined descendant. Learn TF-IDF first; BM25 is the production version.